# Granite 4.2 8B + Ollama + Pydantic  
## Noisy OCR → Maternal Registry Extraction

This notebook is a production-style example of the **LLM portion only**.

It assumes:

- Ollama is already installed.
- The model `granite4.2:8b` is already pulled.
- OCR has already converted the image into text.
- The OCR text may be incomplete, unordered, abbreviated, duplicated, or noisy.

Architecture:

```text
OCR text
   ↓
System prompt
+ 5 few-shot examples
+ target schema
   ↓
granite4.2:8b through Ollama
   ↓
Pydantic / JSON Schema constrained output
   ↓
deterministic validation
   ↓
KNOWN / NEEDS_REVIEW / ILLEGIBLE / NOT_PROVIDED
   ↓
human review where necessary
   ↓
31-column registry row
```

The model performs **document-level schema alignment**, not line-by-line parsing.


## 1. Python dependencies

Ollama itself is already installed on the computer.

The notebook only needs the Python client plus Pydantic and pandas.

Run this cell only if the Python packages are missing.


In [14]:
# Uncomment if needed:
# %pip install -q -U ollama pydantic pandas


## 2. Configuration

In [15]:
MODEL = "granite4.2:3b"

TARGET_COLUMNS = ['id', 'age (years)', 'education level (0=none/primary,1=secondary,2=higher)', 'consanguinity', 'desired pregnancy', 'hypertension history', 'diabetes mellitus', 'gravidity (number)', 'parity (number)', 'abortions (number)', 'living children (number)', 'previous cesarean', 'bmi pregestational (kg/m2)', 'mean systolic bp (mmhg)', 'mean diastolic bp (mmhg)', 'hemoglobin (g/dl)', 'first fasting glucose (mg/dl)', 'proteinuria', 'hiv test result', 'syphilis test result', 'hepatitis c test result', 'gestational age at enrollment (weeks)', 'gestational dm', 'gestational age at birth (weeks)', 'preterm birth', 'type of delivery (0=vaginal,1=cesarean)', 'newborn sex (0=female,1=male)', 'child birth weight (g)', 'head circumference (cm)', 'breastfeeding initiated', 'referral to higher care']

print("Model:", MODEL)
print("Target fields:", len(TARGET_COLUMNS))


Model: granite4.2:3b
Target fields: 31


## 3. Optional: confirm that Ollama sees the model

This does not download or install anything. It only checks the local Ollama model list.


In [16]:
import subprocess

try:
    result = subprocess.run(
        ["ollama", "list"],
        capture_output=True,
        text=True,
        check=True
    )
    print(result.stdout)
except Exception as e:
    print("Could not run `ollama list`:", e)


NAME             ID              SIZE      MODIFIED           
granite4.2:3b    40577dc168a3    2.2 GB    About a minute ago    



## 4. Define the temporary LLM extraction contract

The LLM does **not** directly write the final CSV row.

Every registry field first gets:

- `value`
- `status`
- `evidence`
- `reason`
- `question`

The evidence is important because it lets us audit what OCR fragment caused the extraction.


In [17]:
from enum import Enum
from typing import Any
from pydantic import BaseModel, Field, model_validator


class Status(str, Enum):
    KNOWN = "KNOWN"
    NEEDS_REVIEW = "NEEDS_REVIEW"
    ILLEGIBLE = "ILLEGIBLE"
    NOT_PROVIDED = "NOT_PROVIDED"


class ExtractedField(BaseModel):
    field_name: str
    value: str | int | float | None = None
    status: Status
    evidence: str | None = None
    reason: str | None = None
    question: str | None = None


class ExtractionResponse(BaseModel):
    fields: list[ExtractedField]

    @model_validator(mode="after")
    def validate_registry_fields(self):
        names = [item.field_name for item in self.fields]

        # No duplicates
        if len(names) != len(set(names)):
            raise ValueError("Duplicate registry field names returned.")

        # Exact schema
        missing = set(TARGET_COLUMNS) - set(names)
        unexpected = set(names) - set(TARGET_COLUMNS)

        if missing:
            raise ValueError(f"Missing fields: {sorted(missing)}")
        if unexpected:
            raise ValueError(f"Unexpected fields: {sorted(unexpected)}")

        if len(self.fields) != len(TARGET_COLUMNS):
            raise ValueError(
                f"Expected {len(TARGET_COLUMNS)} fields, got {len(self.fields)}"
            )

        return self


print("Pydantic schema ready.")


Pydantic schema ready.


## 5. Registry coding rules

These rules are supplied to the LLM explicitly rather than relying on its memory.


In [18]:
CODING_RULES = {
    "education level (0=none/primary,1=secondary,2=higher)": {
        "none": 0,
        "primary": 0,
        "secondary": 1,
        "higher": 2
    },

    "binary_fields": {
        "no": 0,
        "yes": 1
    },

    "test_results": {
        "negative": 0,
        "positive": 1
    },

    "type of delivery (0=vaginal,1=cesarean)": {
        "vaginal": 0,
        "cesarean": 1
    },

    "newborn sex (0=female,1=male)": {
        "female": 0,
        "male": 1
    }
}

BINARY_COLUMNS = {
    "consanguinity",
    "desired pregnancy",
    "hypertension history",
    "diabetes mellitus",
    "previous cesarean",
    "proteinuria",
    "hiv test result",
    "syphilis test result",
    "hepatitis c test result",
    "gestational dm",
    "preterm birth",
    "type of delivery (0=vaginal,1=cesarean)",
    "newborn sex (0=female,1=male)",
    "breastfeeding initiated",
    "referral to higher care"
}


## 6. Strong system prompt

The system prompt is stable application configuration.

Important behavior:

- never infer missing medical information,
- never use medical plausibility as evidence,
- search the **whole OCR document**,
- keep exact OCR evidence,
- prefer review over confident guessing.


In [19]:
SYSTEM_PROMPT = """
You are a strict document-extraction engine for maternal registry forms.

You are NOT a clinical decision-support system.
Do not diagnose, recommend treatment, or infer a medical condition.

Your only task is to map noisy OCR text into the supplied registry schema.

GENERAL RULES

1. The OCR may be incomplete, unordered, duplicated, abbreviated, or badly formatted.
2. Do not assume field order.
3. Do not assume one OCR line corresponds to one field.
4. Search the entire OCR document for supporting evidence.
5. Never invent a value.
6. Never derive one registry field from another registry field.
7. Never assign an unlabeled number merely because it is medically plausible.
8. Preserve the exact OCR fragment that supports each extracted value.
9. Use only the coding rules supplied by the application.
10. Return every target field exactly once.

STATUS RULES

KNOWN:
The OCR provides sufficiently clear evidence for the value.

NEEDS_REVIEW:
There is a plausible candidate value, but a meaningful ambiguity remains.
Provide the candidate value when possible and generate one concise clarification question.

ILLEGIBLE:
Relevant OCR text appears to exist, but it cannot be reliably interpreted.

NOT_PROVIDED:
No supporting OCR evidence exists anywhere in the document.

OUTPUT RULES

- Return structured JSON only.
- Follow the supplied JSON Schema exactly.
- For NOT_PROVIDED, value must be null and evidence should be null.
- For KNOWN, evidence should contain the exact supporting OCR fragment.
- Prefer NEEDS_REVIEW rather than confidently guessing.
""".strip()

print(SYSTEM_PROMPT)


You are a strict document-extraction engine for maternal registry forms.

You are NOT a clinical decision-support system.
Do not diagnose, recommend treatment, or infer a medical condition.

Your only task is to map noisy OCR text into the supplied registry schema.

GENERAL RULES

1. The OCR may be incomplete, unordered, duplicated, abbreviated, or badly formatted.
2. Do not assume field order.
3. Do not assume one OCR line corresponds to one field.
4. Search the entire OCR document for supporting evidence.
5. Never invent a value.
6. Never derive one registry field from another registry field.
7. Never assign an unlabeled number merely because it is medically plausible.
8. Preserve the exact OCR fragment that supports each extracted value.
9. Use only the coding rules supplied by the application.
10. Return every target field exactly once.

STATUS RULES

KNOWN:
The OCR provides sufficiently clear evidence for the value.

NEEDS_REVIEW:
There is a plausible candidate value, but a meanin

## 7. Five few-shot examples

These are deliberately focused on **behavior**, especially when the model should *not* guess.

They are much cheaper than fine-tuning and can be changed immediately when a failure pattern is discovered.


In [20]:
FEW_SHOT_EXAMPLES = [
    {
        "input": "g2 p1 a0",
        "lesson": {
            "gravidity (number)": {
                "value": 2,
                "status": "KNOWN",
                "evidence": "g2"
            },
            "parity (number)": {
                "value": 1,
                "status": "KNOWN",
                "evidence": "p1"
            },
            "abortions (number)": {
                "value": 0,
                "status": "KNOWN",
                "evidence": "a0"
            }
        }
    },

    {
        "input": "hiv neg   hcv neg",
        "lesson": {
            "hiv test result": {
                "value": 0,
                "status": "KNOWN",
                "evidence": "hiv neg"
            },
            "hepatitis c test result": {
                "value": 0,
                "status": "KNOWN",
                "evidence": "hcv neg"
            }
        }
    },

    {
        "input": "bp 12O/8O",
        "lesson": {
            "mean systolic bp (mmhg)": {
                "value": 120,
                "status": "NEEDS_REVIEW",
                "evidence": "bp 12O/8O",
                "reason": "OCR may have confused O with 0.",
                "question": "I read the blood pressure as 120/80. Is that correct?"
            },
            "mean diastolic bp (mmhg)": {
                "value": 80,
                "status": "NEEDS_REVIEW",
                "evidence": "bp 12O/8O",
                "reason": "OCR may have confused O with 0.",
                "question": "I read the blood pressure as 120/80. Is that correct?"
            }
        }
    },

    {
        "input": "24.8",
        "lesson": {
            "bmi pregestational (kg/m2)": {
                "value": 24.8,
                "status": "NEEDS_REVIEW",
                "evidence": "24.8",
                "reason": "The number is unlabeled.",
                "question": "Is 24.8 the pregestational BMI?"
            }
        }
    },

    {
        "input": "syph ____",
        "lesson": {
            "syphilis test result": {
                "value": None,
                "status": "ILLEGIBLE",
                "evidence": "syph ____",
                "reason": "The field appears present but the result is unreadable.",
                "question": "What is the syphilis test result?"
            }
        }
    }
]


## 8. Prompt builder

The prompt is assembled by code.

The current patient OCR is inserted only at inference time.


In [21]:
import json


def build_user_prompt(ocr_text: str) -> str:
    return f"""
TARGET REGISTRY FIELDS

{json.dumps(TARGET_COLUMNS, indent=2)}

CODING RULES

{json.dumps(CODING_RULES, indent=2)}

FEW-SHOT BEHAVIOR EXAMPLES

{json.dumps(FEW_SHOT_EXAMPLES, indent=2)}

CURRENT OCR DOCUMENT

---BEGIN OCR---
{ocr_text}
---END OCR---

Extract all target registry fields.

Important:
- Return all {len(TARGET_COLUMNS)} fields exactly once.
- The OCR may be unordered.
- Missing information must remain missing.
- Do not infer unlabeled values merely from plausibility.
- If uncertain, prefer NEEDS_REVIEW.
- Return JSON matching the supplied schema only.
""".strip()


## 9. Example noisy OCR request

Some values are missing and the order is intentionally messy.


In [22]:
ocr_text = '''
pt 201

28
sec

g2 p1 a0
1 living
no cs

wanted preg yes

24.8

bp maybe 12O 8O
hb12.4
fast 86

hiv neg
hcv neg

32 sem

notes:
follow up next month
'''.strip()

print(ocr_text)


pt 201

28
sec

g2 p1 a0
1 living
no cs

wanted preg yes

24.8

bp maybe 12O 8O
hb12.4
fast 86

hiv neg
hcv neg

32 sem

notes:
follow up next month


## 10. Call Granite 4.2 8B through local Ollama

Ollama structured output accepts a JSON Schema through `format`.

Pydantic generates the schema and validates the returned JSON.

`temperature=0` is used for more deterministic extraction.


In [23]:
from ollama import chat


def extract_with_ollama(ocr_text: str) -> ExtractionResponse:
    user_prompt = build_user_prompt(ocr_text)

    response = chat(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        format=ExtractionResponse.model_json_schema(),
        options={
            "temperature": 0
        },
        stream=False
    )

    # Pydantic performs the second validation layer.
    normalized = normalize_extraction(
    response.message.content
    )

    return ExtractionResponse.model_validate(normalized)


## 11. Run the extraction

This is the cell that actually invokes your locally installed `granite4.2:3b`.

The first run can take longer because Ollama has to load the model into memory.


In [24]:
extraction = extract_with_ollama(ocr_text)
print(extraction.model_dump_json(indent=2))


KeyboardInterrupt: 

## 12. Deterministic post-validation

Structured output ensures the JSON shape, but we still enforce application-level constraints in Python.

The model is an **interpreter**, not the application controller.


In [ ]:
def validate_application_rules(extraction: ExtractionResponse) -> None:
    by_name = {item.field_name: item for item in extraction.fields}

    # Education coding
    edu = by_name[
        "education level (0=none/primary,1=secondary,2=higher)"
    ].value
    if edu is not None and edu not in {0, 1, 2}:
        raise ValueError(f"Invalid education code: {edu}")

    # Binary/coded fields
    for col in BINARY_COLUMNS:
        value = by_name[col].value
        if value is not None and value not in {0, 1}:
            raise ValueError(f"Invalid binary code for {col}: {value}")

    # Status / evidence behavior
    for item in extraction.fields:
        if item.status == Status.NOT_PROVIDED:
            if item.value is not None:
                raise ValueError(
                    f"{item.field_name}: NOT_PROVIDED must have value=None"
                )

        if item.status == Status.KNOWN and not item.evidence:
            raise ValueError(
                f"{item.field_name}: KNOWN should contain OCR evidence"
            )


print("Application validator ready.")


## 13. Split fields into accepted / review / missing

This is deterministic routing. The LLM does not decide the UI workflow.


In [ ]:
def route_extraction(extraction: ExtractionResponse):
    accepted = {}
    review = []
    missing = []

    for item in extraction.fields:

        if item.status == Status.KNOWN:
            accepted[item.field_name] = item.value

        elif item.status in {
            Status.NEEDS_REVIEW,
            Status.ILLEGIBLE
        }:
            review.append({
                "field": item.field_name,
                "proposed_value": item.value,
                "evidence": item.evidence,
                "reason": item.reason,
                "question": item.question
            })

        elif item.status == Status.NOT_PROVIDED:
            missing.append(item.field_name)

    return accepted, review, missing


validate_application_rules(extraction)

accepted, review_queue, missing = route_extraction(extraction)

print("Accepted:")
print(json.dumps(accepted, indent=2))

print("\nReview:")
print(json.dumps(review_queue, indent=2))

print("\nMissing:")
print(json.dumps(missing, indent=2))


## 14. Finalize to the exact 31-column registry row

Only after ambiguous values have been confirmed do we discard the temporary LLM metadata.

`confirmed_values` represents corrections / confirmations returned by the phone UI.


In [ ]:
import pandas as pd


def finalize_registry_row(
    extraction: ExtractionResponse,
    confirmed_values: dict | None = None
) -> pd.DataFrame:

    confirmed_values = confirmed_values or {}

    values = {}

    for item in extraction.fields:

        if item.field_name in confirmed_values:
            values[item.field_name] = confirmed_values[item.field_name]

        elif item.status == Status.KNOWN:
            values[item.field_name] = item.value

        else:
            # unresolved / absent values remain null
            values[item.field_name] = None

    return pd.DataFrame(
        [[values[col] for col in TARGET_COLUMNS]],
        columns=TARGET_COLUMNS
    )


confirmed_values = {
    "bmi pregestational (kg/m2)": 24.8,
    "mean systolic bp (mmhg)": 120,
    "mean diastolic bp (mmhg)": 80,
    "gestational age at enrollment (weeks)": 32
}

final_df = finalize_registry_row(
    extraction,
    confirmed_values
)

display(final_df)


## 15. Minimal end-to-end call

Once the pieces are tested, your backend can reduce the workflow to:

```python
extraction = extract_with_ollama(ocr_text)

validate_application_rules(extraction)

accepted, review_queue, missing = route_extraction(extraction)

# send review_queue to phone
# receive confirmed_values from user

final_df = finalize_registry_row(
    extraction,
    confirmed_values
)
```

This is the key production separation:

```text
Granite 4.2 8B
    ↓
interpretation

Pydantic + Python
    ↓
validation and control

Human
    ↓
remaining ambiguity
```


## 16. What to tune before considering fine-tuning

Start by changing these components:

1. **System prompt**
2. **Few-shot examples**
3. **Status definitions**
4. **Coding dictionary**
5. **Pydantic schema**
6. **Post-validation rules**

Keep a synthetic evaluation set containing cases such as:

- clean OCR
- fields in random order
- omitted fields
- `O / 0` OCR confusion
- French / English abbreviations
- unlabeled numbers
- duplicate OCR
- partially illegible fields

Useful metrics:

- field extraction accuracy
- status accuracy
- false extraction rate
- review recall
- schema-valid response rate

For this application, false confident extraction should be treated as more serious than sending an extra field to review.
